In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

# Домашнее задание

Data contains the book rating information. Ratings (Book-Rating) are either explicit, expressed on a scale from 1-10 (higher values denoting higher appreciation), or implicit, expressed by 0.

### 1. Реализовать персональный топ  - принимает на вход возраст и локацию, на выходе персональный топ   - 1 балл (это сделано в другом ноутбуке)

Персональный топ - это топ товаров по похожим возрасту/интересам/локации. Как сделать? Разбить на сегменты по выбраным признакам. Топ делать по книгам с хорошим средним рейтингом.

### 2. На основе метода кластеризации похожих пользователей построить рекомендации (Слайд 27) - 3 балла

Нужно топ-10 рекомендаций с самой высокой оценкой. Считаем среднюю оценку для каждой книги по кластеру и выводим топ-10 книг.

### 3. Совстречаемость - 3 балла

В совстречаемости также учитывать оценки. Вес пары книг встретившихся у пользователя - полусумма их оценок.

### 4. Коллаборативная фильтрация - 3 балла (это сделано в другом ноутбуке)

Коллаборативную фильтрацию реализовывать как на слайде 51 презентации, посоветовав каждому пользователю топ-10 книг с самой высокой оценкой. Сделать рекомендации User-based и Item-based и сравнить.

Если совсем сложно - можно сделать как в семинарской части, поставив оценку "0", если рейтинг < 5 и "1" - в противном случае. Тогда максимум за это - 1 балл. Реализовать U2I и I2I рекомендации.

### Примечание:

Так как пользователей много - можно зафиксировать несколько произвольных и для них уже составлять рекомендации

Работоспособность I2I можно проверять на известных книгах (Гарри Поттер, Властелин Колец, Интервью с вампиром, Код-Да-Винчи, Маленький Принц)

Рейтинг книг обязательно нужно учитывать

Не забываем также предобработать данные - выкинуть выбросы-пользователей и выбросы-книги.

Выводить в качестве рекомендаций лучше названия книг, картинки (если они есть) и соответствующие метрики близости.

In [ ]:
books = pd.read_csv("BX-Books.csv.gz", compression='gzip')

<ipython-input-20-0c30beb66596>:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("BX-Books.csv.gz", compression='gzip')


In [ ]:
interactions = pd.read_csv("BX-Book-Ratings.csv.gz", sep=";", encoding = "ISO-8859-1", compression='gzip')

In [ ]:
interactions = interactions[interactions["Book-Rating"] != 0]

In [ ]:
books_meets = interactions.groupby("ISBN")["User-ID"].count().reset_index().rename(columns={"User-ID": "user_num"})

In [ ]:
user_meets = interactions.groupby("User-ID")["ISBN"].count().reset_index().rename(columns={"ISBN": "books_num"})

In [ ]:
interactions = interactions.merge(books_meets, on=["ISBN"]).merge(user_meets, on=["User-ID"])

In [ ]:
interactions = interactions[(interactions["user_num"] > 5) &
                            (interactions["books_num"] > 5) &
                            (interactions["books_num"] < 200)]

In [ ]:
users = pd.read_csv('BX-Users.csv.gz', delimiter=';', encoding = 'ISO-8859-1')

In [ ]:
interactions = interactions.merge(books[["ISBN", "Image-URL-M", "Book-Title"]].rename(
    columns={"Image-URL-M": "picture_url"}), on=["ISBN"])

In [ ]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

In [ ]:
interactions["product_id"] = le.fit_transform(interactions["ISBN"])
interactions["vid"] = le.fit_transform(interactions["User-ID"])

In [ ]:
interactions.sort_values("vid")

,User-ID,ISBN,Book-Rating,user_num,books_num,picture_url,Book-Title,product_id,vid
88460,8,0002005018,5,9,7,http://images.amazon.com/images/P/0002005018.0...,Clara Callan,0,0
12972,99,0312252617,8,9,8,http://images.amazon.com/images/P/0312252617.0...,Fast Women,1386,1
51337,99,0312261594,8,14,8,http://images.amazon.com/images/P/0312261594.0...,Female Intelligence,1393,1
883,99,0451166892,3,87,8,http://images.amazon.com/images/P/0451166892.0...,The Pillars of the Earth,5415,1
10671,99,0446677450,10,42,8,http://images.amazon.com/images/P/0446677450.0...,"Rich Dad, Poor Dad: What the Rich Teach Their ...",5105,1
...,...,...,...,...,...,...,...,...,...
82725,278851,0439050006,5,9,14,http://images.amazon.com/images/P/0439050006.0...,Captain Underpants and the Wrath of the Wicked...,4214,10958
64753,278851,0440486599,5,12,14,http://images.amazon.com/images/P/0440486599.0...,"Then Again, Maybe I Won't",4607,10958
50989,278854,042516098X,7,64,6,http://images.amazon.com/images/P/042516098X.0...,Hornet's Nest,4019,10959
7420,278854,0553579606,8,52,6,http://images.amazon.com/images/P/0553579606.0...,Ashes to Ashes,6880,10959


In [ ]:
csr_rates = coo_matrix((interactions["Book-Rating"], (interactions["vid"], interactions["product_id"])),
                            shape=(len(set(interactions["vid"])), len(set(interactions["product_id"]))))

In [ ]:
csr_rates

<10960x10712 sparse matrix of type '<class 'numpy.int64'>'
	with 110519 stored elements in COOrdinate format>

2) **Рекомендации на основе кластеризации похожих пользователей**:

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.cluster import BisectingKMeans
import plotly.graph_objs as go
from collections import Counter
scaler = StandardScaler()
tsvd3D = TruncatedSVD(n_components=3, random_state=42)
tsvd_data3D = normalize(scaler.fit_transform(tsvd3D.fit_transform(csr_rates)))

In [ ]:
fig = go.Figure(data=[go.Scatter3d(x=tsvd_data3D[:, 0], y=tsvd_data3D[:, 1], z=tsvd_data3D[:, 2],
                                   mode = 'markers', marker_size = 3,
                                   marker=dict(
                                   colorscale='Viridis', showscale=True))])
fig.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
                                    scene_camera=dict(up=dict(x=0, y=0, z=1),
                                    center=dict(x=0, y=0, z=0),
                                    eye=dict(x=1.25, y=1.25, z=1.25)))
fig.show()

Можно покрутить и визуально оценить, что есть 5 крупных кластеров (с учётом шума). Соответственно запишем в модель BisectingKMeans n_clusters = 5.

In [ ]:
bisectkmeanspp = BisectingKMeans(n_clusters=5, init='k-means++', algorithm='elkan', random_state=42)
clusterB3Dpp = bisectkmeanspp.fit_predict(tsvd_data3D)
tsvdB3Dpp_df = pd.DataFrame(data = tsvd_data3D, columns = ['x', 'y', 'z'])
tsvdB3Dpp_df['cluster'] = clusterB3Dpp
fig2 = go.Figure(data=[go.Scatter3d(x=tsvdB3Dpp_df['x'], y=tsvdB3Dpp_df['y'], z=tsvdB3Dpp_df['z'],
                                   mode = 'markers', marker_size = 3,
                                   marker=dict(color=tsvdB3Dpp_df['cluster'],
                                   colorscale='Viridis', showscale=True))])
fig2.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
                                    scene_camera=dict(up=dict(x=0, y=0, z=1),
                                    center=dict(x=0, y=0, z=0),
                                    eye=dict(x=1.25, y=1.25, z=1.25)))
fig2.show()

In [ ]:
tsvdB3Dpp_df.sort_values('cluster')

,x,y,z,cluster
0,-0.994624,-0.000216,-0.103553,0
6293,-0.959901,-0.074827,-0.270169,0
6292,-0.993387,0.020195,-0.113025,0
6291,-0.989087,0.081949,-0.122440,0
6290,-0.995100,0.008040,-0.098551,0
...,...,...,...,...
9764,0.123787,0.142303,0.982052,4
1961,0.039604,0.035366,0.998589,4
1293,0.373556,-0.282301,0.883608,4
8553,-0.151057,-0.354871,0.922631,4


Далее на основе выполненной кластеризации создаётся словарь пользователей, принадлежащих определённому кластеру.

In [ ]:
df_sorted = tsvdB3Dpp_df.sort_values('cluster')
indexes = df_sorted.index
lst = [indexes[0]]
dictClustUserIds = {}
for i in range(1, len(df_sorted)):
  if df_sorted['cluster'][indexes[i]] != df_sorted['cluster'][indexes[i - 1]]:
    dictClustUserIds[df_sorted['cluster'][indexes[i - 1]]] = lst
    lst = []
  lst.append(indexes[i])
dictClustUserIds[df_sorted['cluster'][indexes[len(df_sorted) - 1]]] = lst

Ниже создаётся словарь оценок. Ключом является vid пользователя, а значение это кортеж (id продукта, оценка). После происходит слияние кластерного словаря и словаря оценок. В итоге получится большой словарь данных, необходимых для построения кластера.

In [ ]:
df_sorted = interactions.sort_values('vid')
indexes = df_sorted.index
lst = []
dictScoreHistory = {}
lst.append((df_sorted['product_id'][indexes[0]], df_sorted['Book-Rating'][indexes[0]]))
for i in range(1, len(df_sorted)):
  if df_sorted['vid'][indexes[i]] != df_sorted['vid'][indexes[i - 1]]:
    dictScoreHistory[df_sorted['vid'][indexes[i - 1]]] = lst
    lst = []
  lst.append((df_sorted['product_id'][indexes[i]], df_sorted['Book-Rating'][indexes[i]]))
dictScoreHistory[df_sorted['vid'][indexes[len(df_sorted) - 1]]] = lst

Слияние кластерного словаря dictClustUserIds с dictScoreHistory:

In [ ]:
clustAppraiserData = {}
for clust, userLst in dictClustUserIds.items():
  clustAppraiserData[clust] = {}
  for appraiser in userLst:
    clustAppraiserData[clust][appraiser] = dictScoreHistory[appraiser]

Теперь мы можем на основе словаря кластеризаций и словаря пользовательских оценок дать пользователю топ 10 рекомендацию по наилучшим оценкам.

Напишем функцию, которая возвращает номер кластера по id оценщика:

In [ ]:
def getClusterByUserId(vid):
  for keyClust, dictAppraiser in clustAppraiserData.items():
    for keyAppraiser in dictAppraiser.keys():
      if keyAppraiser == vid:
        return keyClust

Теперь у нас есть всё для того, чтобы найти топ-10 рекомендаций по id оценщика. Найдём рекомендации для оценщика с vid = 1000.

In [ ]:
lstAllBooksInCluster = []
for id, ratings in clustAppraiserData[getClusterByUserId(1000)].items():
  for rating in ratings:
    lstAllBooksInCluster.append((rating[0], rating[1]))
lstAllBooksInCluster.sort(key=lambda x: x[0])

Нужно убрать книги, которые имеют 1-5 оценки в кластере, так как на основе такого малого количества оценок рекомендовать было бы очень странно.

In [ ]:
lstBooksFiltered = []
counterBooks = Counter(bookScore[0] for bookScore in lstAllBooksInCluster)
lstBooksFiltered = [x for x in lstAllBooksInCluster if counterBooks[x[0]] > 5]

Для каждой книги считаем среднюю оценку

In [ ]:
lst = []
bookMeanRatingDict = {}
lst.append(lstBooksFiltered[0][1])
for i in range(1, len(lstBooksFiltered)):
  if lstBooksFiltered[i][0] != lstBooksFiltered[i - 1][0]:
    bookMeanRatingDict[lstBooksFiltered[i][0]] = 1.0 * sum(lst)/len(lst)
    lst = []
  lst.append(lstBooksFiltered[i][1])
bookMeanRatingDict[lstBooksFiltered[len(lstBooksFiltered) - 1][0]] = 1.0 * sum(lst)/len(lst)

Находим топ 10 товаров по средним оценкам для пользователя с vid = 1000.

In [ ]:
bookMeanCounter = Counter(bookMeanRatingDict)
bestProductIds = []
for avgRating in bookMeanCounter.most_common(10):
  bestProductIds.append(avgRating)

Выводим топ 10.

In [ ]:
for i in bestProductIds:
  for j, k in interactions[["product_id", "Book-Title"]].drop_duplicates().values:
    if j == i[0]:
      print("idx:", i[0], "\tAvg. Rating:", round(i[1], 2), "\tBook Title:", k)

idx: 196 	Avg. Rating: 10.0 	Book Title: Wicked : The Life and Times of the Wicked Witch of the West
idx: 902 	Avg. Rating: 9.71 	Book Title: Of Mice and Men (Penguin Great Books of the 20th Century)
idx: 1247 	Avg. Rating: 9.71 	Book Title: Life of Pi
idx: 818 	Avg. Rating: 9.62 	Book Title: On the Road
idx: 8678 	Avg. Rating: 9.54 	Book Title: I Do (But I Don't)
idx: 706 	Avg. Rating: 9.5 	Book Title: Catherine, Called Birdy (Trophy Newbery)
idx: 4840 	Avg. Rating: 9.5 	Book Title: A Yellow Raft in Blue Water
idx: 8523 	Avg. Rating: 9.5 	Book Title: Fall On Your Knees (Oprah #45)
idx: 2145 	Avg. Rating: 9.44 	Book Title: Billy Straight : A Novel
idx: 284 	Avg. Rating: 9.43 	Book Title: Brave New World


In [ ]:
len(interactions.sort_values('vid')['vid'])

110519

3) **Рекомендации на основе совстречаемости**

Хоть дубликатов быть не может, так как юзер не может поставить оценку два раза, но лишним не будет, мало ли чего:

In [ ]:
df_tmp = interactions[['vid', 'product_id', 'Book-Rating']].drop_duplicates(subset=['vid', 'product_id'], keep=False)

Для каждого vid сделаем список кортежей, где будет храниться id книги и оценка:

In [ ]:
df_tmp['book_rate'] = df_tmp[['product_id', 'Book-Rating']].apply(tuple, axis=1)
df_tmp.drop(['product_id', 'Book-Rating'], axis = 1, inplace = True)
df_tmp = df_tmp.groupby(["vid"])["book_rate"].apply(list).reset_index()
product_num = [len(lst) for lst in df_tmp['book_rate']]#подсчитываем количество оценок для каждого пользователя
df_tmp['marks_num'] = product_num
df_tmp = df_tmp[df_tmp['marks_num'] > 1] #убираем пользователей, которые делали оценку лишь однажды
df_tmp.head()

,vid,book_rate,marks_num
1,1,"[(5415, 3), (5105, 10), (1386, 8), (1393, 8), ...",5
2,2,"[(7386, 10), (5064, 9), (5014, 9), (5628, 8), ...",6
3,3,"[(6529, 10), (10379, 9), (10526, 8), (10354, 8)]",4
4,4,"[(1684, 9), (415, 7), (1743, 7), (1715, 9), (2...",16
5,5,"[(415, 7), (421, 8), (9916, 6), (9068, 8), (11...",8


Воспользуемся реализацией совстречаемости с семинара, немного изменив принцип подсчёта пар (учитываем среднюю их оценку)

In [ ]:
prodPairsDict = {}
for lst in df_tmp['book_rate']:
  for i in range(len(lst)):
    for j in range(len(lst)):
      if (i != j):
        try:
          prodPairsDict[str(lst[i][0]) + '_' + str(lst[j][0])] += 1.0 * (lst[i][1] + lst[j][1]) / 2
        except:
          prodPairsDict[str(lst[i][0]) + '_' + str(lst[j][0])] = 1.0 * (lst[i][1] + lst[j][1]) / 2
recommendList = []
for pair, measure in prodPairsDict.items():
  if (measure > 16.0):
    #если пара товаров с высокой средней оценкой (я поставил от 8: 8*2 = 16) встречаются более одного раза, то добавляем в список
    recommendList.append(pair.split('_') + [measure])
recommendDf = pd.DataFrame(recommendList, columns=["book1", "book2", "measure"])


In [ ]:
def getRecommendations(i):
  recommendations = recommendDf[recommendDf["book1"] == str(i)].sort_values("measure", ascending=False).head(10)
  tmp_df = interactions.drop_duplicates(subset=['product_id'])
  print("\nДля товара:", str(tmp_df[tmp_df['product_id'] == i]['Book-Title'].values[0]))
  print("\nБудут следующие рекомендации:\n")
  for (book2, measure) in zip(recommendations["book2"].values, recommendations["measure"].values.astype(int)):
    print(str(tmp_df[tmp_df['product_id'] == int(book2)]['Book-Title'].values[0]), "| Measure:", measure)

In [ ]:
getRecommendations(1000)


Для товара: Riding in Cars With Boys: Confessions of a Bad Girl Who Makes Good

Будут следующие рекомендации:

The Catcher in the Rye | Measure: 23
The Lovely Bones: A Novel | Measure: 23
Charlotte's Web (Trophy Newbery) | Measure: 17
I Don't Know How She Does It: The Life of Kate Reddy, Working Mother | Measure: 16


Видно, что рекомендации на основе совстречаемости более менее работают.

Книжке про женское, наример (Riding in Cars With Boys: Confessions of a Bad Girl Who Makes Good), соответствуют книги про плюс минус то же самое.

Логично, что если два товара встречаются в паре у двух пользователей, и при этом хорошо оцениваются ими, то скорее всего эти товары будут принадлежать какой-то одной тематике, которые можно было бы порекомендовать.